In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# ----------------------------------------------------
# 1. LOAD WEIGHTED RETURN & RISK FROM EDA
# ----------------------------------------------------
mu = np.load("mu_annual.npy")              # weighted annual returns
sigma = np.load("sigma_annual.npy")        # weighted annual SD (risk)
tickers = pd.read_csv("tickers.csv")["0"].tolist()

n = len(mu)

In [13]:
# ----------------------------------------------------
# 2. OPTIMIZATION: MAX RETURN SUBJECT TO MAX RISK
# ----------------------------------------------------
def max_return_under_risk(max_risk):
    """
    Maximize sum(w_i * mu_i)
    subject to:
        sum(w_i * sigma_i) <= max_risk        (risk constraint)
        sum(w_i) = 1                          (budget)
        w_i >= 0                              (long-only)
    """

    # linprog MINIMIZES, so we minimize -return to maximize return
    c = -mu

    # Risk constraint: sigma^T w <= max_risk
    A_ub = [sigma]
    b_ub = [max_risk]

    # Budget constraint: sum(w) = 1
    A_eq = [np.ones(n)]
    b_eq = [1.0]

    bounds = [(0, 0.25) for _ in range(n)] # long-only with maximum weight of 25%

    result = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not result.success:
        raise RuntimeError(f"Optimization failed: {result.message}")

    w = result.x

    port_return = np.dot(w, mu)
    port_risk = np.dot(w, sigma)   # linear, weighted risk measure

    return w, port_return, port_risk

In [11]:
# ----------------------------------------------------
# 3. DEFINE RISK TOLERANCE FOR EACH PROFILE
# ----------------------------------------------------
risk_limits = {
    "Aggressive":   0.45,   # allow up to 45% annual SD
    "Balanced":     0.25,   # up to 25%
    "Conservative": 0.18    # up to 18%
}

In [14]:
# 4. SOLVE FOR EACH PROFILE
# ----------------------------------------------------
for profile, max_risk in risk_limits.items():
    w, ret, risk = max_return_under_risk(max_risk)

    df = pd.DataFrame({
        "Ticker": tickers,
        "Weight": w
    }).sort_values("Weight", ascending=False)

    print("\n==========================================")
    print(profile.upper())
    print("==========================================")
    print(f"Max Return:      {ret * 100:.2f}%")
    print(f"Weighted Risk:   {risk * 100:.2f}% (limit = {max_risk * 100:.1f}%)\n")

    print(df[df["Weight"] > 0.001])

    df.to_csv(f"{profile.lower()}_weighted_portfolio.csv", index=False)

print("\nDone. Portfolios saved as CSV files.")


AGGRESSIVE
Max Return:      67.92%
Weighted Risk:   41.68% (limit = 45.0%)

   Ticker  Weight
10     GE    0.25
22   NVDA    0.25
6     CEG    0.25
17    LLY    0.25

BALANCED
Max Return:      41.65%
Weighted Risk:   25.00% (limit = 25.0%)

   Ticker    Weight
10     GE  0.250000
31    WMT  0.250000
30   WELL  0.250000
4   BRK-B  0.180908
22   NVDA  0.069092

CONSERVATIVE
Max Return:      16.58%
Weighted Risk:   18.00% (limit = 18.0%)

   Ticker    Weight
4   BRK-B  0.250000
14    JNJ  0.250000
27     SO  0.250000
31    WMT  0.212143
23     PG  0.037857

Done. Portfolios saved as CSV files.


In [24]:
def max_return_low_corr(max_risk, max_corr_score):
    """
    Maximize w^T mu
    Subject to:
        w^T sigma <= max_risk
        w^T corr_score <= max_corr_score
        sum(w) = 1
        0 <= w_i <= 0.25
    """

    c = -mu  # maximize return

    # Two linear inequality constraints
    A_ub = np.vstack([sigma, corr_score])
    b_ub = [max_risk, max_corr_score]

    A_eq = [np.ones(n)]
    b_eq = [1.0]

    bounds = [(0, 0.75)] * n

    result = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not result.success:
        raise RuntimeError(result.message)

    w = result.x
    port_return = np.dot(w, mu)
    port_risk = np.dot(w, sigma)
    port_corr = np.dot(w, corr_score)

    return w, port_return, port_risk, port_corr

In [25]:
risk_limits = {
    "Aggressive_LC": 0.45,
    "Balanced_LC":   0.25,
    "Conservative_LC": 0.18,
}

corr_limits = {
    "Aggressive_LC": 0.55,
    "Balanced_LC":   0.45,
    "Conservative_LC": 0.35,
}

for profile in risk_limits:

    max_risk = risk_limits[profile]
    max_corr_score = corr_limits[profile]

    w, ret, risk, corr_s = max_return_low_corr(max_risk, max_corr_score)

    df = pd.DataFrame({"Ticker": tickers, "Weight": w}).sort_values("Weight", ascending=False)

    print("\n=========================================")
    print(profile)
    print("=========================================")
    print(f"Return:            {ret:.2%}")
    print(f"Risk:              {risk:.2%} (limit {max_risk})")
    print(f"CorrScore (w·k):   {corr_s:.3f} (limit {max_corr_score})")

    print(df[df["Weight"] > 0.001])
    df.to_csv(f"{profile}_lowcorr_portfolio.csv", index=False)


Aggressive_LC
Return:            82.34%
Risk:              45.00% (limit 0.45)
CorrScore (w·k):   0.474 (limit 0.55)
   Ticker    Weight
22   NVDA  0.665914
10     GE  0.334086

Balanced_LC
Return:            41.62%
Risk:              25.00% (limit 0.25)
CorrScore (w·k):   0.450 (limit 0.45)
   Ticker    Weight
31    WMT  0.468957
10     GE  0.364527
30   WELL  0.166516

Conservative_LC
Return:            14.44%
Risk:              18.00% (limit 0.18)
CorrScore (w·k):   0.350 (limit 0.35)
   Ticker    Weight
14    JNJ  0.750000
1    ABBV  0.149319
31    WMT  0.098303
10     GE  0.002378
